# Atmospheric (FSO) Channel Simulation

This notebook demonstrates the `atmospheric` environment provided by OWCM.


In [ ]:
import sys
sys.path.append('../')
from owcm import Channel
import numpy as np
import matplotlib.pyplot as plt


## 1. Build the Channel
We use the `mvp` recipe which includes Kim attenuation, Gamma-Gamma turbulence, and Pointing errors.


In [ ]:
ch = Channel.environment('atmospheric').defaults('mvp').build(
    distance_m=1000, 
    wavelength_nm=1550, 
    pt_w=0.1, 
    visibility_km=2.5,
    gg_alpha=4.0, gg_beta=1.9,
    jitter_sigma_m=0.02, wz_eq_m=0.1
)
print(ch.describe())


## 2. Generate Samples and Plot
We simulate 5000 samples of the received power to visualize fading.


In [ ]:
N = 5000
power_samples = ch.received_power_samples(N)
plt.figure(figsize=(10, 4))
plt.plot(power_samples, alpha=0.6, label='Inst. Power')
plt.axhline(np.mean(power_samples), color='r', linestyle='--', label='Mean')
plt.title('Received Power with Turbulence & Pointing')
plt.xlabel('Sample Index')
plt.ylabel('Power (W)')
plt.legend()
plt.grid(True)
plt.show()


## 3. Outage Probability vs Threshold


In [ ]:
thresholds = np.logspace(-9, -3, 20) # 1nW to 1mW
outages = [ch.outage(th, n=10000) for th in thresholds]
plt.figure(figsize=(6, 6))
plt.loglog(thresholds, outages, 'o-')
plt.xlabel('Threshold (W)')
plt.ylabel('Outage Probability')
plt.title('Outage Probability Curve')
plt.grid(True, which='both', linestyle='--')
plt.show()


## 4. Received Power vs Distance
Analyzing how average received power decays with distance.


In [ ]:
d_range = np.linspace(100, 5000, 20)
pr_avg = []
for d in d_range:
    ch_temp = Channel.environment('atmospheric').defaults('mvp').build(
        distance_m=d, wavelength_nm=1550, pt_w=0.1, visibility_km=2.0
    )
    pr_avg.append(ch_temp.received_power())

plt.figure(figsize=(8, 5))
plt.semilogy(d_range, pr_avg, 'b-o')
plt.xlabel('Distance (m)')
plt.ylabel('Average Received Power (W)')
plt.title('Received Power vs Distance (Vis=2km)')
plt.grid(True, which='both', linestyle='--')
plt.show()
